## Model Comparison

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [46]:
import pandas as pd
from src.data_utils import load_processed
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.model import save_model

In [5]:
df=load_processed('model_ready.csv')

### Train-Validation Split

In [11]:
X=df.drop(columns=['TARGET'])
y=df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    stratify=y, 
                                                    test_size=0.2, 
                                                    random_state=42)

### Random Forest

In [34]:
cat_col = df.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_col)
    ],
    remainder='passthrough'
)

pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(max_depth=8, n_estimators=150, random_state=42))
])

In [35]:
pipe_rf.fit(X_train, y_train)

y_pred_rf = pipe_rf.predict_proba(X_test)[:,1]
y_train_rf = pipe_rf.predict_proba(X_train)[:,1]

rfv_auc = roc_auc_score(y_test, y_pred_rf)
rft_auc = roc_auc_score(y_train, y_train_rf)

print(f'Random Forest Train AUC: {rft_auc}')
print(f'Random Forest Validation AUC: {rfv_auc}')

Random Forest Train AUC: 0.7819612023271805
Random Forest Validation AUC: 0.7459102999000716


#### Results

Train AUC: 0.7819    
Validation AUC: 0.7459    

### Gradient Boosting

In [38]:
pipe_gb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))
])

In [39]:
pipe_gb.fit(X_train, y_train)

y_pred_gb = pipe_gb.predict_proba(X_test)[:,1]
y_train_gb = pipe_gb.predict_proba(X_train)[:,1]

gbt_auc = roc_auc_score(y_train, y_train_gb)
gbv_auc = roc_auc_score(y_test, y_pred_gb)

print(f'Gradient Boosting Train AUC: {gbt_auc}')
print(f'Gradient Boosting Validation AUC: {gbv_auc}')

Gradient Boosting Train AUC: 0.7713974765449164
Gradient Boosting Validation AUC: 0.7654058119596737


#### Results

Train AUC: 0.7713    
Validation AUC: 0.7654    

In [51]:
save_model(pipe_gb, 'gb_v1_auc_0_765.joblib')

✅ Model saved at: C:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\models\gb_v1_auc_0_765.joblib


### Conclusion

Gradient Boosting achieved the highest validation AUC (0.7654) with minimal overfitting (gap = 0.006).

Logistic regression performed strongly indicating linear signal.

Random forest showed high varience.

Therefore, as of now Gradient boosting selected as the final model.